# Task 3: soft mixture-of-experts (Kaggle run)

Task 3 starts from the trained Task 2 models, so they must be attached first:
1. On kaggle.com: Datasets -> New Dataset -> upload `task2_outputs.zip` (from the Task 2 run), keep it private.
2. In this notebook: Add Input -> Your Datasets -> attach that dataset.
3. Same settings as before: GPU on, Internet on, `WANDB_API_KEY` secret attached
   (and `GITHUB_TOKEN` if the repository is private).

In [ ]:
# 1. Secrets -> environment variables
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "genai-a1"

REPO = "aleenababar04/genai-assignment-1"
try:
    token = secrets.get_secret("GITHUB_TOKEN")   # only needed for a private repo
    REPO_URL = f"https://{token}@github.com/{REPO}.git"
except Exception:
    REPO_URL = f"https://github.com/{REPO}.git"


In [ ]:
# 2. Get the code and install the few packages Kaggle does not ship
%cd /kaggle/working
!rm -rf genai-assignment-1
!git clone -q $REPO_URL genai-assignment-1
%cd genai-assignment-1
!git log --oneline -1
!pip install -q optuna wandb onnx onnxruntime pytorch-msssim python-dotenv

import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn the GPU on in the notebook settings."


In [ ]:
# 3. Copy the trained Task 2 checkpoints from the attached dataset into checkpoints/
import glob, shutil

os.makedirs("checkpoints", exist_ok=True)
needed = ["task2_classifier.pt", "task2_specialist_salt_pepper.pt",
          "task2_specialist_blur.pt", "task2_specialist_occlusion.pt"]
for file_name in needed:
    found = glob.glob(f"/kaggle/input/**/{file_name}", recursive=True)
    assert found, f"{file_name} not found: attach the Task 2 outputs dataset (see the top of this notebook)."
    shutil.copy(found[0], "checkpoints/")
    print("copied", found[0])


In [ ]:
# 4. Download Oxford-IIIT Pet and build the 128x128 cache (split checked against the repo)
!python -m src.data.prepare_pet
!python -m pytest -q


In [ ]:
# 5. Optuna search over the joint fine-tuning stage, then full training -> checkpoints/task3_moe.pt
!python -m src.training.train_moe --mode optuna
!python -m src.training.train_moe --mode final


In [ ]:
# 6. Test evaluation with gate analysis, and ONNX export of the complete pipeline
!python -m src.evaluation.eval_moe
!python -m src.export.export_onnx --task task3


In [ ]:
# 7. Collect everything that must go back into the repository.
#    Download task3_outputs.zip from the notebook's Output panel.
!mkdir -p /kaggle/working/task3_outputs/configs /kaggle/working/task3_outputs/checkpoints
!cp configs/task3_moe_best.yaml /kaggle/working/task3_outputs/configs/
!cp checkpoints/task3_moe.pt /kaggle/working/task3_outputs/checkpoints/
!cp -r optuna_studies models report/results report/figures /kaggle/working/task3_outputs/
%cd /kaggle/working
!zip -qr task3_outputs.zip task3_outputs
!ls -lh task3_outputs.zip
